In [7]:
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from webdriver_manager.chrome import ChromeDriverManager


def setup_driver(headless: bool = False):
    chrome_options = Options()
    chrome_options.add_argument("--start-maximized")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    if headless:
        chrome_options.add_argument("--headless=new")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=chrome_options
    )
    return driver


def crawl_gangnamunni_categories():
    driver = setup_driver(headless=False)
    wait = WebDriverWait(driver, 10)

    url = "https://www.gangnamunni.com/events?categoryId=422"  # 예시: 리프팅
    driver.get(url)

    # 메인 컨텐츠 로딩 대기
    time.sleep(3)

    data = []

    # ========== 1뎁스 카테고리 찾기 (셀렉터 강화) ==========
    try:
        # nav 안의 ul.flex.flex-row.px-0 요소 대기
        nav_ul = wait.until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "nav ul.flex.flex-row.px-0")
            )
        )
    except Exception:
        print("[ERROR] nav ul.flex.flex-row.px-0 를 찾지 못했습니다.")
        # 디버깅용: 페이지 소스를 저장해보고 싶다면 아래 주석 해제
        # with open("gangnamunni_page_source.html", "w", encoding="utf-8") as f:
        #     f.write(driver.page_source)
        driver.quit()
        return pd.DataFrame()

    # nav 안에서 클릭 가능한 모든 후보들 (a, button, div)
    depth1_candidates = nav_ul.find_elements(
        By.XPATH,
        ".//a | .//button | .//div"
    )

    depth1_list = []
    for idx, el in enumerate(depth1_candidates):
        try:
            text = el.text.strip()
            if text and el.is_displayed():
                depth1_list.append((idx, text))
        except Exception:
            continue

    print("1뎁스 카테고리 목록:")
    for _, name in depth1_list:
        print(" -", name)

    if not depth1_list:
        print("[WARN] 1뎁스 카테고리를 하나도 찾지 못했습니다.")
        driver.quit()
        return pd.DataFrame()

    # ========== 각 1뎁스 탭 클릭 → 2뎁스 수집 ==========
    for depth1_idx, depth1_name in depth1_list:
        print(f"\n▶ 1뎁스 클릭: {depth1_name}")

        # 매 루프마다 nav_ul / 후보들 다시 찾기 (stale element 방지)
        try:
            nav_ul = driver.find_element(By.CSS_SELECTOR, "nav ul.flex.flex-row.px-0")
            curr_candidates = nav_ul.find_elements(
                By.XPATH,
                ".//a | .//button | .//div"
            )
        except Exception as e:
            print(f"[WARN] {depth1_name} 탐색 중 nav 영역을 다시 찾지 못함: {e}")
            continue

        if depth1_idx >= len(curr_candidates):
            print(f"[WARN] {depth1_name} (index {depth1_idx}) 스킵 - 후보 개수 변경됨")
            continue

        current_depth1 = curr_candidates[depth1_idx]

        try:
            driver.execute_script("arguments[0].click();", current_depth1)
        except Exception as e:
            print(f"[WARN] {depth1_name} 클릭 실패: {e}")
            continue

        time.sleep(1.5)

        # --- 2뎁스 영역 찾기 (hide-scrollbar + gap-4 조합) ---
        depth2_names = []
        try:
            # main 내부에 hide-scrollbar 클래스를 가진 div
            container_divs = driver.find_elements(
                By.XPATH,
                "//main//div[contains(@class, 'hide-scrollbar')]"
            )

            for cont in container_divs:
                try:
                    ul = cont.find_element(
                        By.XPATH,
                        ".//ul[contains(@class, 'flex') and contains(@class, 'gap-4')]"
                    )
                    elems = ul.find_elements(
                        By.XPATH,
                        ".//a | .//button | .//div"
                    )
                    for e2 in elems:
                        text = e2.text.strip()
                        if text:
                            depth2_names.append(text)
                except Exception:
                    continue

        except Exception as e:
            print(f"[INFO] {depth1_name} : 2뎁스 영역 못 찾음 - {e}")
            depth2_names = []

        # 중복 제거
        depth2_names = list(dict.fromkeys(depth2_names))

        if not depth2_names:
            print(f"  └ 2뎁스 없음 → 1뎁스만 기록")
            data.append({"depth1": depth1_name, "depth2": None})
        else:
            print(f"  └ 2뎁스 {len(depth2_names)}개 발견")
            for d2 in depth2_names:
                data.append({
                    "depth1": depth1_name,
                    "depth2": d2
                })

    # ========== DataFrame 정리 & 엑셀 저장 ==========
    df = pd.DataFrame(data).drop_duplicates().reset_index(drop=True)
    print("\n[미리보기]")
    print(df.head())

    output_path = "gangnamunni_categories_depth1_depth2.xlsx"
    df.to_excel(output_path, index=False)
    print(f"\n엑셀 저장 완료: {output_path}")

    driver.quit()
    return df


if __name__ == "__main__":
    crawl_gangnamunni_categories()

[ERROR] nav ul.flex.flex-row.px-0 를 찾지 못했습니다.
